In [1]:
from pathlib import Path
import pandas as pd


data_path = Path("Data/PetFinder/train.csv")

if not data_path.exists():
    data_path = Path("../Data/Petfinder/train.csv")

df = pd.read_csv(data_path)

print("Shape:", df.shape)
display(df.head())

Shape: (14993, 24)


,Type,Name,Age,Breed1,Breed2,Gender,Color1,Color2,Color3,MaturitySize,...,Health,Quantity,Fee,State,RescuerID,VideoAmt,Description,PetID,PhotoAmt,AdoptionSpeed
0,2,Nibble,3,299,0,1,1,7,0,1,...,1,1,100,41326,8480853f516546f6cf33aa88cd76c379,0,Nibble is a 3+ month old ball of cuteness. He ...,86e1089a3,1.0,2
1,2,No Name Yet,1,265,0,1,1,2,0,2,...,1,1,0,41401,3082c7125d8fb66f7dd4bff4192c8b14,0,I just found it alone yesterday near my apartm...,6296e909a,2.0,0
2,1,Brisco,1,307,0,1,2,7,0,2,...,1,1,0,41326,fa90fa5b1ee11c86938398b60abc32cb,0,Their pregnant mother was dumped by her irresp...,3422e4906,7.0,3
3,1,Miko,4,307,0,2,1,2,0,2,...,1,1,150,41401,9238e4f44c71a75282e62f7136c6b240,0,"Good guard dog, very alert, active, obedience ...",5842f1ff5,8.0,2
4,1,Hunter,1,307,0,1,1,0,0,2,...,1,1,0,41326,95481e953f8aed9ec3d16fc4509537e8,0,This handsome yet cute boy is up for adoption....,850a43f90,3.0,2


In [2]:
y = df['AdoptionSpeed'].astype("int32")
X_text = df["Description"].fillna("").astype(str)
numeric_cols = ['Age' , 'Quantity' , 'Fee' , "VideoAmt" ,'PhotoAmt']
categorical_cols = [
    "Type", "Breed1", "Breed2", "Gender",
    "Color1", "Color2", "Color3",
    "MaturitySize", "FurLength",
    "Vaccinated", "Dewormed", "Sterilized",
    "Health", "State",
]

X_structured = df[numeric_cols + categorical_cols].copy()

In [3]:
from sklearn.model_selection import train_test_split

(
    train_text, val_text,
    train_structured, val_structured,
    y_train, y_val,
) = train_test_split(
    X_text,
    X_structured,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Text:", train_text.shape, val_text.shape)
print("Structured:", train_structured.shape, val_structured.shape)
print("Target:", y_train.shape, y_val.shape)

Text: (11994,) (2999,)
Structured: (11994, 19) (2999, 19)
Target: (11994,) (2999,)


In [6]:
from sklearn.preprocessing import StandardScaler , OneHotEncoder
from sklearn.compose import ColumnTransformer 
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer 

numeric_pipeline = Pipeline([
    ("imputer" , SimpleImputer(strategy = 'median')),
    ("scaler" , StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer' , SimpleImputer(strategy = 'most_frequent')),
    ('onehot' , OneHotEncoder(handle_unknown = 'ignore', sparse_output = False))
])

preprocessor = ColumnTransformer([
    ('numeric' , numeric_pipeline , numeric_cols),
    ('categorical' , categorical_pipeline , categorical_cols)
])

train_structured_ready = preprocessor.fit_transform(train_structured).astype('float32')
val_structured_ready = preprocessor.transform(val_structured).astype('float32')

In [7]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

train_text_ready = np.asarray(
    train_text, dtype=str
).reshape(-1, 1)

val_text_ready = np.asarray(
    val_text, dtype=str
).reshape(-1, 1)

text_vectorizer = layers.TextVectorization(
    max_tokens=10_000,
    output_mode="int",
    output_sequence_length=150,
)

text_vectorizer.adapt(train_text_ready)

print("Description:", train_text_ready[0, 0])
print("\nToken IDs:")
print(text_vectorizer(train_text_ready[:1]).numpy())

print("\nVocabulary size:", text_vectorizer.vocabulary_size())

Description: These 3 puppies was rescued from a mechanic shop in sentul, apparently we managed to save 4 out of the initially 6 puppies. Also these 3 are really energetic and very playful, you will love their enthusiasm and adorableness...^^, hence if anyone is interested in adopting and providing a good home for them, please feel free to whats app or call(Mohan) or(Charlie) *i would prefer whatsapp as i am a student because i will be in and out classes often

Token IDs:
[[ 125   75   92   30   79   52    4 4575  465   12 4479 2535   28  737
     3  599  114  130   11    5 1609  162   92  122  125   75   13  148
   540    2   17   59   18   26   58  110 5171    2    1  644   23  164
     6   56   12  300    2 1943    4   44   15    7   27   21  323  184
     3 1306 1087   37    1    1    8   93  476  206   31    8  115    4
   956  132    8   26   19   12    2  130    1  810    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    

In [13]:
#tokens to embeddings 
text_input = keras.Input(shape = (1,) , dtype = 'string' , name='text_input')

tokens = text_vectorizer(text_input)

embedded = layers.Embedding(
    input_dim=text_vectorizer.vocabulary_size(),
    output_dim=64,
    mask_zero=True,
    name="word_embeddings",
)(tokens)

rnn_output = layers.SimpleRNN(64 ,name = 'text_rnn')(embedded)
text_features = layers.Dense(32,activation ='relu' , name = 'text_features')(rnn_output)

In [14]:
text_branch = keras.Model(
    inputs=text_input,
    outputs=text_features,
    name="text_branch",
)

text_branch.summary()

Model: "text_branch"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ text_input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_vectorization  │ (None, 150)       │          0 │ text_input[0][0]  │
│ (TextVectorization) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ word_embeddings     │ (None, 150, 64)   │    640,000 │ text_vectorizati… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, 150)       │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_rnn            │ (None, 64)        │      8,256 │ word_embeddings[… │
│ (SimpleRNN)         │                   │            │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_features       │ (None, 32)        │      2,080 │ text_rnn[0][0]    │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 650,336 (2.48 MB)

 Trainable params: 650,336 (2.48 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
#structured branch 

structured_input = keras.Input(shape = (train_structured_ready.shape[1],), dtype = 'float32' , name = 'structured_input')
structured_features = layers.Dense(32,activation = 'relu')(structured_input)

print(structured_input.shape)
print(structured_features.shape)

(None, 356)
(None, 32)


In [18]:
combined = layers.Concatenate(name = 'combine_inputs')([text_features , structured_features])
hidden = layers.Dense(32, activation = 'relu' , name = 'combined_featues')(combined)

prediction = layers.Dense(5 , activation ='softmax')(hidden)

model = keras.Model(inputs = [text_input , structured_input] , outputs = prediction , name = 'petfinder')

model.summary()

Model: "petfinder"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ text_input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_vectorization  │ (None, 150)       │          0 │ text_input[0][0]  │
│ (TextVectorization) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ word_embeddings     │ (None, 150, 64)   │    640,000 │ text_vectorizati… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, 150)       │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_rnn            │ (None, 64)        │      8,256 │ word_embeddings[… │
│ (SimpleRNN)         │                   │            │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ structured_input    │ (None, 356)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_features       │ (None, 32)        │      2,080 │ text_rnn[0][0]    │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 32)        │     11,424 │ structured_input… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ combine_inputs      │ (None, 64)        │          0 │ text_features[0]… │
│ (Concatenate)       │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ combined_featues    │ (None, 32)        │      2,080 │ combine_inputs[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 5)         │        165 │ combined_featues… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 664,005 (2.53 MB)

 Trainable params: 664,005 (2.53 MB)

 Non-trainable params: 0 (0.00 B)

In [24]:

text_input = keras.Input(
    shape=(1,), dtype="string", name="text_input"
)

tokens = text_vectorizer(text_input)

embedded = layers.Embedding(
    input_dim=text_vectorizer.vocabulary_size(),
    output_dim=32,
    mask_zero=True,
)(tokens)

text_features = layers.SimpleRNN(
    16,
    dropout=0.3,
)(embedded)

text_features = layers.Dense(16, activation="relu")(text_features)


structured_input = keras.Input(
    shape=(train_structured_ready.shape[1],),
    name="structured_input",
)

structured_features = layers.Dense(
    32, activation="relu"
)(structured_input)

structured_features = layers.Dropout(0.3)(structured_features)

# Combine branches
combined = layers.Concatenate()([
    text_features, structured_features
])

hidden = layers.Dense(16, activation="relu")(combined)
hidden = layers.Dropout(0.3)(hidden)

prediction = layers.Dense(
    5, activation="softmax", name="adoption_speed"
)(hidden)

model = keras.Model(
    inputs=[text_input, structured_input],
    outputs=prediction,
)

In [25]:
from pathlib import Path

Path("Models").mkdir(exist_ok=True)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),

    keras.callbacks.ModelCheckpoint(
        filepath="Models/petfinder_best.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1,
    ),
]

In [26]:
#converting the description to tensors for both train and val
import tensorflow as tf
train_text_tensor = tf.constant(
    train_text.to_numpy(dtype=str).reshape(-1, 1),
    dtype=tf.string,
)

val_text_tensor = tf.constant(
    val_text.to_numpy(dtype=str).reshape(-1, 1),
    dtype=tf.string,
)

history = model.fit(
    x={
        "text_input": train_text_tensor,
        "structured_input": train_structured_ready,
    },
    y=y_train.to_numpy(),
    validation_data=(
        {
            "text_input": val_text_tensor,
            "structured_input": val_structured_ready,
        },
        y_val.to_numpy(),
    ),
    epochs=30,
    batch_size=32,
    callbacks=callbacks,
)

Epoch 1/30
375/375 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.2956 - loss: 1.4885
Epoch 1: val_loss improved from None to 1.42399, saving model to Models/petfinder_best.keras

Epoch 1: finished saving model to Models/petfinder_best.keras
375/375 ━━━━━━━━━━━━━━━━━━━━ 17s 33ms/step - accuracy: 0.2956 - loss: 1.4885 - val_accuracy: 0.3505 - val_loss: 1.4240 - learning_rate: 0.0010
Epoch 2/30
375/375 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.3478 - loss: 1.4223
Epoch 2: val_loss improved from 1.42399 to 1.39940, saving model to Models/petfinder_best.keras

Epoch 2: finished saving model to Models/petfinder_best.keras
375/375 ━━━━━━━━━━━━━━━━━━━━ 12s 32ms/step - accuracy: 0.3478 - loss: 1.4223 - val_accuracy: 0.3655 - val_loss: 1.3994 - learning_rate: 0.0010
Epoch 3/30
375/375 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.3873 - loss: 1.3665
Epoch 3: val_loss improved from 1.39940 to 1.37790, saving model to Models/petfinder_best.keras

Epoch 3: finished saving model to Models/